# Clase 07 — Git y GitHub para analistas de datos

> Notebook maestro. Las celdas `!git ...` se ejecutan de verdad: crean un repo de juguete
> en una carpeta temporal, asi la clase se puede repetir cuantas veces haga falta sin ensuciar nada.

## Por que un analista necesita git

Todos hemos visto esta carpeta:

```
analisis.ipynb
analisis_v2.ipynb
analisis_v2_FINAL.ipynb
analisis_v2_FINAL_ok.ipynb
analisis_v2_FINAL_ok_ESTE_SI.ipynb
```

Los tres problemas que eso genera:

1. **Nadie sabe cual es el bueno.** Ni siquiera quien los creo, dos semanas despues.
2. **Nadie sabe que cambio entre uno y otro.** El nombre no lo dice.
3. **Si borras el bueno, se acabo.** No hay vuelta atras.

Git resuelve los tres: un solo archivo, un historial con mensajes, y todo recuperable.

## Los 4 estados de un archivo

```
   working dir          staging area          repositorio         remoto
   (tu carpeta)         (el "carrito")        (.git local)        (GitHub)
        |                     |                     |                  |
        |----- git add ------>|                     |                  |
        |                     |---- git commit ---->|                  |
        |                     |                     |---- git push --->|
        |<-------------------------- git pull -------------------------|
```

Si entiendes ese dibujo, entiendes el 90% de git. El otro 10% son las ramas.

---
## 0. Verificar la instalacion

Si esto falla: https://git-scm.com/download/win

In [ ]:
!git --version

## 1. Configuracion inicial

Se hace **una sola vez por maquina**. El nombre y el correo quedan grabados en cada commit
que hagas para siempre — usa el mismo correo de tu cuenta de GitHub.

In [ ]:
# Descomentar y poner tus datos reales:
# !git config --global user.name "Tu Nombre"
# !git config --global user.email "tu.correo@ejemplo.com"

!git config --global --list

## 2. Un repo de juguete

Trabajamos en una carpeta temporal. Nada de lo que pase aqui toca tus archivos.

In [ ]:
import tempfile, os
from pathlib import Path

DEMO = Path(tempfile.mkdtemp()) / "demo-analisis"
DEMO.mkdir(parents=True)
os.chdir(DEMO)
print("Trabajando en:", DEMO)

In [ ]:
!git init -b main
# Config local del repo demo: asi la practica funciona aunque todavia no hayas
# hecho el 'git config --global'. En tus repos de verdad usa el global.
!git config user.name "Alumno Demo"
!git config user.email "demo@ejemplo.com"
!git status

`git status` es el comando que mas vas a usar. **Ante la duda, `git status`.**
Te dice donde estas parado y, casi siempre, cual es el siguiente comando.

## 3. El ciclo diario: add -> commit

In [ ]:
Path("analisis.py").write_text("import pandas as pd\n\nventas = pd.read_csv('ventas.csv')\n", encoding="utf-8")
!git status --short

`??` significa *untracked*: git ve el archivo pero no lo esta siguiendo todavia.

In [ ]:
!git add analisis.py
!git status --short

`A` significa *added to staging*: esta en el carrito, listo para el commit.

In [ ]:
!git commit -m "Carga inicial de ventas"
!git log --oneline

### El mensaje de commit

Un mensaje util responde **que cambio y por que**, no "cambios".

| Malo | Bueno |
|---|---|
| `cambios` | `Corrige el separador del CSV: era ; no ,` |
| `update` | `Agrega filtro de ventas anuladas al reporte mensual` |
| `asdf` | `Elimina duplicados antes del merge para no inflar el total` |

Regla practica: si el mensaje sirve para cualquier commit, no sirve para ninguno.

Formato: **imperativo, presente, una linea, menos de 72 caracteres.**
"Agrega", no "Agregue" ni "Agregando".

## 4. Ver que cambio: `git diff`

In [ ]:
Path("analisis.py").write_text(
    "import pandas as pd\n\nventas = pd.read_csv('ventas.csv', sep=';', decimal=',')\n",
    encoding="utf-8")
!git diff

`-` es lo que se fue, `+` lo que entro. Esto es lo que el nombre del archivo
nunca te iba a decir.

In [ ]:
!git add analisis.py
!git commit -m "Corrige el separador del CSV: era ; no ,"
!git log --oneline

## 5. `.gitignore` — lo que NUNCA se sube

Tres categorias que jamas van a un repo:

1. **Credenciales.** Contrasenas, tokens, cadenas de conexion. Una vez subidas quedan en el historial aunque las borres despues.
2. **Datos.** Los CSV pesados y, sobre todo, cualquier dato personal de clientes. Un repo publico con datos de clientes es un incidente, no un descuido.
3. **Basura generada.** `.ipynb_checkpoints/`, `__pycache__/`, `.venv/`.

In [ ]:
Path(".gitignore").write_text(
    "# datos\n*.csv\n*.xlsx\ndata/\n\n"
    "# credenciales\n.env\nconfig_local.py\n\n"
    "# basura\n.ipynb_checkpoints/\n__pycache__/\n.venv/\n",
    encoding="utf-8")

Path("ventas.csv").write_text("id;importe\nV01;100\n", encoding="utf-8")
Path(".env").write_text("DB_PASSWORD=supersecreto123\n", encoding="utf-8")

!git status --short

`ventas.csv` y `.env` **no aparecen**. Git los esta ignorando. Eso es exactamente lo que queremos.

In [ ]:
!git add .gitignore
!git commit -m "Agrega .gitignore para datos y credenciales"
!git log --oneline

> **Importante:** `.gitignore` solo funciona sobre archivos que git **todavia no sigue**.
> Si ya commiteaste una contrasena, agregarla al `.gitignore` no la borra del historial.
> Ahi la unica salida real es **rotar la credencial**.

## 6. Ramas

Una rama es una linea de trabajo paralela. Sirven para probar algo sin romper lo que ya funciona.

Para un analista, el caso tipico: "el gerente quiere ver el reporte con otra segmentacion,
pero el actual ya esta aprobado".

In [ ]:
!git switch -c prueba-segmentacion
!git branch

In [ ]:
Path("analisis.py").write_text(
    "import pandas as pd\n\n"
    "ventas = pd.read_csv('ventas.csv', sep=';', decimal=',')\n"
    "por_segmento = ventas.groupby('segmento')['importe'].sum()\n",
    encoding="utf-8")
!git add analisis.py
!git commit -m "Agrega corte por segmento"
!git log --oneline

In [ ]:
# Volver a main: el archivo vuelve a como estaba, intacto
!git switch main
print(Path("analisis.py").read_text(encoding="utf-8"))

El cambio no desaparecio: sigue vivo en la otra rama. `main` nunca se rompio.

In [ ]:
!git merge prueba-segmentacion
print(Path("analisis.py").read_text(encoding="utf-8"))
!git log --oneline

## 7. Un conflicto, a proposito

Un conflicto pasa cuando dos ramas tocan **la misma linea** del mismo archivo.
Git no adivina cual gana: te lo pregunta. No es un error, es una pregunta.

In [ ]:
!git switch -c precio-dolares
Path("analisis.py").write_text("MONEDA = 'USD'\n", encoding="utf-8")
!git add -A && git commit -q -m "Cambia la moneda a dolares"

!git switch main
Path("analisis.py").write_text("MONEDA = 'PEN'\n", encoding="utf-8")
!git add -A && git commit -q -m "Fija la moneda en soles"

!git merge precio-dolares

In [ ]:
print(Path("analisis.py").read_text(encoding="utf-8"))

Git deja las dos versiones marcadas:

```
<<<<<<< HEAD
MONEDA = 'PEN'          <- lo que hay en tu rama actual
=======
MONEDA = 'USD'          <- lo que viene de la otra rama
>>>>>>> precio-dolares
```

**Resolverlo es editar el archivo a mano**, dejar la version buena, borrar las 3 lineas de
marcas, y commitear. No hay comando magico: la decision es tuya, git solo te la muestra.

In [ ]:
Path("analisis.py").write_text("MONEDA = 'PEN'\n", encoding="utf-8")
!git add analisis.py
!git commit -m "Resuelve conflicto: se mantiene soles como moneda base"
!git log --oneline --graph --all

## 8. GitHub

Hasta aqui todo fue **local**: el historial vive en la carpeta `.git` de tu maquina.
GitHub es una copia en la nube. Sirve para:

- que no se pierda si se muere el disco
- trabajar con otros
- **tener portafolio** — para un analista junior esto vale mas que el CV

### El flujo, una vez creado el repo en github.com

```bash
git remote add origin https://github.com/TU-USUARIO/TU-REPO.git
git push -u origin main
```

De ahi en adelante, todos los dias:

```bash
git pull      # traer lo que hicieron otros ANTES de empezar
# ... trabajas ...
git add .
git commit -m "mensaje util"
git push      # subir lo tuyo
```

> **`git pull` antes de empezar, siempre.** El 90% de los conflictos feos nacen de
> trabajar sobre una version vieja.

## 9. Notebooks en git: el problema y la solucion

Un `.ipynb` es un JSON que **guarda los outputs adentro**. Consecuencias:

- Ejecutas el notebook sin cambiar ni una linea de codigo -> git dice que cambio.
- Un `df.head()` de 500 filas o un grafico en base64 hacen diffs de miles de lineas ilegibles.
- Si el output tiene datos de clientes, los estas publicando sin darte cuenta.

**La regla:** limpiar los outputs antes de commitear.

```bash
# manual: Kernel -> Restart & Clear Output, guardar, commitear
# automatico:
pip install nbstripout
nbstripout --install     # se engancha al repo y limpia solo en cada commit
```

## 10. Chuleta

| Necesito | Comando |
|---|---|
| Donde estoy parado | `git status` |
| Que cambie | `git diff` |
| Guardar un avance | `git add .` y `git commit -m "..."` |
| Ver el historial | `git log --oneline --graph` |
| Deshacer cambios NO commiteados de un archivo | `git restore archivo.py` |
| Sacar algo del carrito (sin perder el cambio) | `git restore --staged archivo.py` |
| Cambiar el mensaje del ultimo commit | `git commit --amend -m "nuevo mensaje"` |
| Nueva rama y saltar a ella | `git switch -c nombre-rama` |
| Volver a main | `git switch main` |
| Traer lo del remoto | `git pull` |
| Subir lo mio | `git push` |
| Recuperar un archivo borrado (ya commiteado) | `git restore --source=HEAD archivo.py` |

**Lo que NO hay que hacer nunca sin entenderlo:** `git reset --hard`, `git push --force`.
Los dos borran trabajo de forma dificil de recuperar.

In [ ]:
# Volver al directorio del curso
os.chdir(Path.home())
print("Demo terminada. La carpeta temporal se puede borrar:", DEMO)